In [ ]:
from bisect import bisect_left, bisect_right
from functools import cache

# Dynamic Programming

---
# 1) Introduction

DP = recursion over **overlapping subproblems** + remembering answers. Every DP question is
answered by writing down four things before any code:

| Step | Question to answer | Example (House Robber) |
|---|---|---|
| **State** | what does `dp[i]` (or `dp[i][j]`) mean, in words? | best loot from houses `0..i` |
| **Transition** | how is a state built from smaller states? | `dp[i] = max(dp[i-1], dp[i-2] + nums[i])` |
| **Base case** | smallest states with known answers | `dp[0] = nums[0]` |
| **Answer** | which state(s) hold the final result? | `dp[n-1]` |

Two ways to implement the same recurrence:

- **Top-down (memoised recursion)** — write the recursion, add `@cache`. Easiest to get
  right in an interview; start here.
- **Bottom-up (tabulation)** — fill a table in dependency order. Lets you **compress
  space** when each row depends only on the previous one (`O(n²)` → `O(n)` memory).

Family map for recognising DP questions:

| Family | Cue | Template state |
|---|---|---|
| 1D linear | "ways to reach / best up to index i" | `dp[i]` |
| Knapsack | pick items under a capacity / target sum | `dp[capacity]` (0/1: iterate capacity **downwards**; unbounded: upwards) |
| Two sequences | compare / align two strings | `dp[i][j]` over prefixes |
| Grid | paths moving right/down | `dp[r][c]` |
| Intervals | split / merge a range, palindromes, burst | `dp[l][r]` by increasing length |
| State machine | holding / not holding / cooldown | `dp[i][state]` |

**How to use each question block.** Read the *Problem*, then try to write the four lines
above yourself before reading the *Idea*. Tiers: ★★★ Blind 75, ★★ NeetCode 150 / Grind 169 /
Top Interview 150, ★ extended.

---
# 2) One-Dimensional DP

## 2.1 Fibonacci-style recurrences

### 2.1.1 [70. Climbing Stairs](https://leetcode.com/problems/climbing-stairs/) — ★★★ · Easy

**Problem.** Climb `n` steps taking 1 or 2 at a time; how many distinct ways?  
**Signal.** Ways to reach step `i` = ways to reach `i−1` + ways to reach `i−2` → Fibonacci.  
**Idea.** The last move onto step `i` is either a 1-step from `i−1` or a 2-step from `i−2`. Only the two previous values are needed, so keep two variables.  
**Cost.** O(n) time · O(1) space

In [ ]:
def climb_stairs(n: int) -> int:
    """Number of 1/2-step ways to climb n stairs."""
    prev, cur = 1, 1
    for _ in range(n - 1):
        prev, cur = cur, prev + cur
    return cur


assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(45) == 1836311903
print(climb_stairs(5))

### 2.1.2 [509. Fibonacci Number](https://leetcode.com/problems/fibonacci-number/) — ★ · Easy

**Problem.** Compute `F(n)` with `F(0) = 0`, `F(1) = 1`.  
**Signal.** The canonical overlapping-subproblem recursion.  
**Idea.** Naive recursion is exponential because `F(n−2)` is recomputed inside `F(n−1)`. Iterate with two variables instead.  
**Cost.** O(n) time · O(1) space

In [ ]:
def fib(n: int) -> int:
    """n-th Fibonacci number."""
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


assert [fib(n) for n in range(8)] == [0, 1, 1, 2, 3, 5, 8, 13]
print(fib(30))

### 2.1.3 [1137. N-th Tribonacci Number](https://leetcode.com/problems/n-th-tribonacci-number/) — ★ · Easy

**Problem.** `T(n) = T(n−1) + T(n−2) + T(n−3)` with `T(0)=0, T(1)=1, T(2)=1`.  
**Signal.** Fibonacci with three terms.  
**Idea.** Roll three variables forward.  
**Cost.** O(n) time · O(1) space

In [ ]:
def tribonacci(n: int) -> int:
    """n-th Tribonacci number."""
    a, b, c = 0, 1, 1
    for _ in range(n):
        a, b, c = b, c, a + b + c
    return a


assert tribonacci(4) == 4
assert tribonacci(25) == 1389537
print(tribonacci(25))

### 2.1.4 [746. Min Cost Climbing Stairs](https://leetcode.com/problems/min-cost-climbing-stairs/) — ★★ · Easy

**Problem.** Pay `cost[i]` to step on stair `i`, then climb 1 or 2; start at index 0 or 1. Minimum cost to get past the top?  
**Signal.** `dp[i]` = cheapest cost to stand on step `i`.  
**Idea.** `dp[i] = cost[i] + min(dp[i−1], dp[i−2])`; the top is reached from either of the last two steps, so answer `min(dp[n−1], dp[n−2])`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def min_cost_climbing_stairs(cost: list[int]) -> int:
    """Cheapest way past the last stair."""
    a, b = cost[0], cost[1]
    for c in cost[2:]:
        a, b = b, c + min(a, b)
    return min(a, b)


assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
print(min_cost_climbing_stairs([10, 15, 20]))

### 2.1.5 [198. House Robber](https://leetcode.com/problems/house-robber/) — ★★★ · Medium

**Problem.** Rob houses in a row without robbing two adjacent ones; maximise loot.  
**Signal.** Take-or-skip at each index → `dp[i] = max(dp[i−1], dp[i−2] + nums[i])`.  
**Idea.** Either skip house `i` (keep the best up to `i−1`) or rob it (best up to `i−2` plus its value). Keep the two previous bests.  
**Cost.** O(n) time · O(1) space

In [ ]:
def rob(nums: list[int]) -> int:
    """Max loot from non-adjacent houses."""
    prev, cur = 0, 0
    for x in nums:
        prev, cur = cur, max(cur, prev + x)
    return cur


assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
print(rob([2, 7, 9, 3, 1]))

### 2.1.6 [213. House Robber II](https://leetcode.com/problems/house-robber-ii/) — ★★★ · Medium

**Problem.** Same as House Robber, but houses are in a circle (first and last are adjacent).  
**Signal.** Break the circle: the answer excludes either the first or the last house.  
**Idea.** Run House Robber on `nums[1:]` and on `nums[:-1]`; take the max. Handle a single house separately.  
**Cost.** O(n) time · O(1) space

In [ ]:
def rob_circle(nums: list[int]) -> int:
    """Max loot when houses form a circle."""
    if len(nums) == 1:
        return nums[0]
    return max(rob(nums[1:]), rob(nums[:-1]))


assert rob_circle([2, 3, 2]) == 3
assert rob_circle([1, 2, 3, 1]) == 4
assert rob_circle([1, 2, 3]) == 3
print(rob_circle([2, 3, 2]))

### 2.1.7 [740. Delete and Earn](https://leetcode.com/problems/delete-and-earn/) — ★ · Medium

**Problem.** Taking value `x` earns `x` but deletes every `x−1` and `x+1`. Maximise points.  
**Signal.** Bucket by value → House Robber over the value axis.  
**Idea.** `points[v] = v × count(v)`. Choosing `v` forbids `v−1` and `v+1`, exactly the adjacency rule of House Robber on the array `points[0..max]`.  
**Cost.** O(n + max) time

In [ ]:
def delete_and_earn(nums: list[int]) -> int:
    """Max points under the delete-neighbours rule."""
    points = [0] * (max(nums) + 1)
    for x in nums:
        points[x] += x
    return rob(points)


assert delete_and_earn([3, 4, 2]) == 6
assert delete_and_earn([2, 2, 3, 3, 3, 4]) == 9
print(delete_and_earn([2, 2, 3, 3, 3, 4]))

### 2.1.8 [91. Decode Ways](https://leetcode.com/problems/decode-ways/) — ★★★ · Medium

**Problem.** Count decodings of a digit string where `A=1 … Z=26`.  
**Signal.** Last step consumes one digit (`1–9`) or two digits (`10–26`) → Fibonacci with conditions.  
**Idea.** `dp[i]` = decodings of the prefix of length `i`. Add `dp[i−1]` if `s[i−1] != '0'`, add `dp[i−2]` if `s[i−2:i]` is between `10` and `26`.  
**Cost.** O(n) time · O(1) space  
**Trap.** A `0` can only be decoded as part of `10` or `20`.

In [ ]:
def num_decodings(s: str) -> int:
    """Number of ways to decode s."""
    prev, cur = 1, 1 if s[0] != "0" else 0
    for i in range(1, len(s)):
        nxt = 0
        if s[i] != "0":
            nxt += cur
        if 10 <= int(s[i - 1 : i + 1]) <= 26:
            nxt += prev
        prev, cur = cur, nxt
    return cur


assert num_decodings("12") == 2
assert num_decodings("226") == 3
assert num_decodings("06") == 0
assert num_decodings("2101") == 1
print(num_decodings("226"))

### 2.1.9 [2140. Solving Questions With Brainpower](https://leetcode.com/problems/solving-questions-with-brainpower/) — ★ · Medium

**Problem.** Question `i` gives `points` but makes you skip the next `brainpower` questions. Max points?  
**Signal.** Decision at each index with a jump → DP from the back.  
**Idea.** `dp[i] = max(dp[i+1], points[i] + dp[i + brainpower + 1])`. Filling right to left means every referenced state is already known.  
**Cost.** O(n) time · O(n) space

In [ ]:
def most_points(questions: list[list[int]]) -> int:
    """Best total score with skip penalties."""
    n = len(questions)
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        points, skip = questions[i]
        dp[i] = max(dp[i + 1], points + dp[min(n, i + skip + 1)])
    return dp[0]


assert most_points([[3, 2], [4, 3], [4, 4], [2, 5]]) == 5
assert most_points([[1, 1], [2, 2], [3, 3], [4, 4], [5, 5]]) == 7
print(most_points([[3, 2], [4, 3], [4, 4], [2, 5]]))

### 2.1.10 [983. Minimum Cost For Tickets](https://leetcode.com/problems/minimum-cost-for-tickets/) — ★ · Medium

**Problem.** Travel days in a year; 1-, 7- and 30-day passes with given costs. Minimum total cost?  
**Signal.** `dp[day]` = cheapest cost to cover all travel days up to `day`.  
**Idea.** On a non-travel day, `dp[d] = dp[d−1]`. On a travel day, try each pass ending today: `cost + dp[d − duration]` (clamped at 0).  
**Cost.** O(365) time

In [ ]:
def mincost_tickets(days: list[int], costs: list[int]) -> int:
    """Cheapest pass combination covering all days."""
    travel = set(days)
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        if d not in travel:
            dp[d] = dp[d - 1]
            continue
        dp[d] = min(dp[max(0, d - span)] + c for span, c in zip((1, 7, 30), costs))
    return dp[-1]


assert mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11
assert mincost_tickets([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [2, 7, 15]) == 17
print(mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]))

## 2.2 Kadane and subarray DP

### 2.2.1 [53. Maximum Subarray](https://leetcode.com/problems/maximum-subarray/) — ★★★ · Medium

**Problem.** Largest sum of a non-empty contiguous subarray.  
**Signal.** Best subarray *ending at i* → Kadane's algorithm.  
**Idea.** The best sum ending at `i` either extends the best sum ending at `i−1` or starts fresh at `nums[i]`: `cur = max(x, cur + x)`. The answer is the max over all `i`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_sub_array(nums: list[int]) -> int:
    """Largest contiguous subarray sum."""
    best = cur = nums[0]
    for x in nums[1:]:
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


assert max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_sub_array([1]) == 1
assert max_sub_array([-3, -1, -2]) == -1
print(max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

### 2.2.2 [152. Maximum Product Subarray](https://leetcode.com/problems/maximum-product-subarray/) — ★★★ · Medium

**Problem.** Largest product of a contiguous subarray.  
**Signal.** Kadane for products — a negative number swaps max and min.  
**Idea.** Track both the max and min product ending at `i`. Multiplying by a negative turns the smallest (most negative) product into the largest. A zero resets both.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_product(nums: list[int]) -> int:
    """Largest contiguous subarray product."""
    best = hi = lo = nums[0]
    for x in nums[1:]:
        candidates = (x, hi * x, lo * x)
        hi, lo = max(candidates), min(candidates)
        best = max(best, hi)
    return best


assert max_product([2, 3, -2, 4]) == 6
assert max_product([-2, 0, -1]) == 0
assert max_product([-2, 3, -4]) == 24
print(max_product([2, 3, -2, 4]))

### 2.2.3 [918. Maximum Sum Circular Subarray](https://leetcode.com/problems/maximum-sum-circular-subarray/) — ★★ · Medium

**Problem.** Maximum subarray sum where the array is circular.  
**Signal.** A wrapping subarray = total − (minimum subarray in the middle).  
**Idea.** Run Kadane for both the max and the min subarray. Answer is `max(max_sub, total − min_sub)`, except when every element is negative (then `total − min_sub` would be the empty array) — return `max_sub`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_subarray_sum_circular(nums: list[int]) -> int:
    """Largest circular subarray sum."""
    cur_max = cur_min = 0
    best_max, best_min = nums[0], nums[0]
    for x in nums:
        cur_max = max(x, cur_max + x)
        cur_min = min(x, cur_min + x)
        best_max = max(best_max, cur_max)
        best_min = min(best_min, cur_min)
    return best_max if best_max < 0 else max(best_max, sum(nums) - best_min)


assert max_subarray_sum_circular([1, -2, 3, -2]) == 3
assert max_subarray_sum_circular([5, -3, 5]) == 10
assert max_subarray_sum_circular([-3, -2, -3]) == -2
print(max_subarray_sum_circular([5, -3, 5]))

### 2.2.4 [413. Arithmetic Slices](https://leetcode.com/problems/arithmetic-slices/) — ★ · Medium

**Problem.** Count contiguous subarrays of length ≥ 3 with constant difference.  
**Signal.** `dp[i]` = number of arithmetic slices *ending* at `i`.  
**Idea.** If `nums[i] − nums[i−1] == nums[i−1] − nums[i−2]`, every slice ending at `i−1` extends, plus one new slice of length 3: `dp[i] = dp[i−1] + 1`; otherwise 0. Sum all `dp[i]`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def number_of_arithmetic_slices(nums: list[int]) -> int:
    """Count arithmetic subarrays of length >= 3."""
    total = cur = 0
    for i in range(2, len(nums)):
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
            total += cur
        else:
            cur = 0
    return total


assert number_of_arithmetic_slices([1, 2, 3, 4]) == 3
assert number_of_arithmetic_slices([1]) == 0
print(number_of_arithmetic_slices([1, 2, 3, 4, 6, 8, 10]))

## 2.3 Word and partition DP

### 2.3.1 [139. Word Break](https://leetcode.com/problems/word-break/) — ★★★ · Medium

**Problem.** Can `s` be segmented into dictionary words?  
**Signal.** `dp[i]` = prefix of length `i` is segmentable.  
**Idea.** `dp[i]` is true if some `j < i` has `dp[j]` true and `s[j:i]` in the dictionary. Only check `j` within the longest word length of `i`.  
**Cost.** O(n · L) checks, each O(L) for slicing

In [ ]:
def word_break(s: str, word_dict: list[str]) -> bool:
    """True if s splits into dictionary words."""
    words = set(word_dict)
    longest = max(map(len, words))
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        for j in range(max(0, i - longest), i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    return dp[-1]


assert word_break("leetcode", ["leet", "code"])
assert word_break("applepenapple", ["apple", "pen"])
assert not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])
print(word_break("leetcode", ["leet", "code"]))

### 2.3.2 [140. Word Break II](https://leetcode.com/problems/word-break-ii/) — ★ · Hard

**Problem.** Return every sentence formed by segmenting `s` into dictionary words.  
**Signal.** Word Break with memoised lists of suffix sentences.  
**Idea.** `sentences(i)` = all segmentations of `s[i:]`. For each word that matches at `i`, prefix it to every sentence of `sentences(i + len(word))`. Memoisation prevents recomputing shared suffixes.  
**Cost.** exponential output in the worst case

In [ ]:
def word_break_ii(s: str, word_dict: list[str]) -> list[str]:
    """All segmentations of s into words."""
    words = set(word_dict)

    @cache
    def sentences(i: int) -> list[str]:
        if i == len(s):
            return [""]
        out = []
        for j in range(i + 1, len(s) + 1):
            if s[i:j] in words:
                for rest in sentences(j):
                    out.append(s[i:j] + (" " + rest if rest else ""))
        return out

    return sentences(0)


res = word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"])
assert sorted(res) == ["cat sand dog", "cats and dog"]
assert word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []
print(res)

### 2.3.3 [279. Perfect Squares](https://leetcode.com/problems/perfect-squares/) — ★★ · Medium

**Problem.** Fewest perfect squares summing to `n`.  
**Signal.** Unbounded knapsack / coin change with coins `1, 4, 9, …`.  
**Idea.** `dp[i] = 1 + min(dp[i − s])` over squares `s ≤ i`.  
**Cost.** O(n √n) time · O(n) space

In [ ]:
def num_squares(n: int) -> int:
    """Fewest squares summing to n."""
    squares = [k * k for k in range(1, int(n**0.5) + 1)]
    dp = [0] + [n] * n
    for i in range(1, n + 1):
        for s in squares:
            if s > i:
                break
            dp[i] = min(dp[i], dp[i - s] + 1)
    return dp[n]


assert num_squares(12) == 3
assert num_squares(13) == 2
print(num_squares(12))

### 2.3.4 [343. Integer Break](https://leetcode.com/problems/integer-break/) — ★ · Medium

**Problem.** Split `n ≥ 2` into at least two positive integers maximising their product.  
**Signal.** `dp[i] = max over j of max(j, dp[j]) × max(i−j, dp[i−j])`.  
**Idea.** Each part may be kept whole or split further, hence `max(j, dp[j])`. (The math shortcut is to use as many 3s as possible.)  
**Cost.** O(n²) time

In [ ]:
def integer_break(n: int) -> int:
    """Max product of a split of n into >= 2 parts."""
    dp = [0, 1] + [0] * (n - 1)
    for i in range(2, n + 1):
        for j in range(1, i // 2 + 1):
            dp[i] = max(dp[i], max(j, dp[j]) * max(i - j, dp[i - j]))
    return dp[n]


assert integer_break(2) == 1
assert integer_break(10) == 36
print(integer_break(10))

### 2.3.5 [96. Unique Binary Search Trees](https://leetcode.com/problems/unique-binary-search-trees/) — ★ · Medium

**Problem.** Number of structurally unique BSTs with keys `1..n`.  
**Signal.** Choose the root, multiply the counts of left and right subtrees → Catalan numbers.  
**Idea.** With root `r`, the left subtree has `r − 1` keys and the right `n − r`: `G(n) = Σ G(r−1) · G(n−r)`.  
**Cost.** O(n²) time

In [ ]:
def num_trees(n: int) -> int:
    """Count of unique BSTs with n keys."""
    g = [1] + [0] * n
    for nodes in range(1, n + 1):
        for root in range(1, nodes + 1):
            g[nodes] += g[root - 1] * g[nodes - root]
    return g[n]


assert num_trees(3) == 5
assert num_trees(1) == 1
print([num_trees(n) for n in range(1, 8)])

## 2.4 Longest increasing subsequence

### 2.4.1 [300. Longest Increasing Subsequence](https://leetcode.com/problems/longest-increasing-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest strictly increasing subsequence.  
**Signal.** O(n²) DP or O(n log n) patience sorting.  
**Idea.** Keep `tails[k]` = the smallest possible tail of an increasing subsequence of length `k+1`. For each `x`, binary search the first tail ≥ `x` and replace it (or append). `tails` stays sorted, and its length is the answer.  
**Cost.** O(n log n) time · O(n) space

In [ ]:
def length_of_lis(nums: list[int]) -> int:
    """Length of the longest strictly increasing subsequence."""
    tails: list[int] = []
    for x in nums:
        i = bisect_left(tails, x)
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x
    return len(tails)


assert length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert length_of_lis([0, 1, 0, 3, 2, 3]) == 4
assert length_of_lis([7, 7, 7, 7]) == 1
print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))

### 2.4.2 [673. Number of Longest Increasing Subsequence](https://leetcode.com/problems/number-of-longest-increasing-subsequence/) — ★ · Medium

**Problem.** How many longest increasing subsequences are there?  
**Signal.** O(n²) LIS that also counts ways.  
**Idea.** `length[i]` and `count[i]` describe LIS ending at `i`. For each `j < i` with `nums[j] < nums[i]`: a longer length replaces the count, an equal length adds to it. Sum counts of indices achieving the max length.  
**Cost.** O(n²) time

In [ ]:
def find_number_of_lis(nums: list[int]) -> int:
    """Number of longest increasing subsequences."""
    n = len(nums)
    length, count = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if length[j] + 1 > length[i]:
                    length[i], count[i] = length[j] + 1, count[j]
                elif length[j] + 1 == length[i]:
                    count[i] += count[j]
    best = max(length)
    return sum(c for l, c in zip(length, count) if l == best)


assert find_number_of_lis([1, 3, 5, 4, 7]) == 2
assert find_number_of_lis([2, 2, 2, 2, 2]) == 5
print(find_number_of_lis([1, 3, 5, 4, 7]))

### 2.4.3 [354. Russian Doll Envelopes](https://leetcode.com/problems/russian-doll-envelopes/) — ★ · Hard

**Problem.** Max number of envelopes that nest (both width and height strictly larger).  
**Signal.** Sort by width ascending, height *descending*, then LIS on heights.  
**Idea.** Sorting heights descending within equal widths prevents two same-width envelopes from being chained by the LIS. The problem then reduces to 1D LIS.  
**Cost.** O(n log n)

In [ ]:
def max_envelopes(envelopes: list[list[int]]) -> int:
    """Longest nesting chain of envelopes."""
    envelopes = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    return length_of_lis([h for _, h in envelopes])


assert max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
assert max_envelopes([[1, 1], [1, 1], [1, 1]]) == 1
print(max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]))

### 2.4.4 [368. Largest Divisible Subset](https://leetcode.com/problems/largest-divisible-subset/) — ★ · Medium

**Problem.** Largest subset where every pair `(a, b)` has `a % b == 0` or `b % a == 0`.  
**Signal.** Sort, then LIS where "increasing" means "divisible by the previous".  
**Idea.** After sorting, a chain `a₁ | a₂ | … | a_k` is pairwise divisible. `dp[i]` = longest chain ending at `i`; keep a parent pointer to rebuild it.  
**Cost.** O(n²)

In [ ]:
def largest_divisible_subset(nums: list[int]) -> list[int]:
    """A largest pairwise-divisible subset."""
    nums = sorted(nums)
    n = len(nums)
    dp, parent = [1] * n, [-1] * n
    for i in range(n):
        for j in range(i):
            if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                dp[i], parent[i] = dp[j] + 1, j
    i = dp.index(max(dp))
    out = []
    while i != -1:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]


assert largest_divisible_subset([1, 2, 3]) in ([1, 2], [1, 3])
assert largest_divisible_subset([1, 2, 4, 8]) == [1, 2, 4, 8]
print(largest_divisible_subset([1, 2, 4, 8]))

### 2.4.5 [1027. Longest Arithmetic Subsequence](https://leetcode.com/problems/longest-arithmetic-subsequence/) — ★ · Medium

**Problem.** Length of the longest arithmetic subsequence.  
**Signal.** State must include the common difference: `dp[i][d]`.  
**Idea.** `dp[i]` is a dict mapping difference `d` → longest arithmetic subsequence ending at `i` with difference `d`. For each `j < i`, `dp[i][d] = dp[j].get(d, 1) + 1` where `d = nums[i] − nums[j]`.  
**Cost.** O(n²) time and space

In [ ]:
def longest_arith_seq_length(nums: list[int]) -> int:
    """Longest arithmetic subsequence length."""
    dp: list[dict[int, int]] = [{} for _ in nums]
    best = 2
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            dp[i][d] = dp[j].get(d, 1) + 1
            best = max(best, dp[i][d])
    return best


assert longest_arith_seq_length([3, 6, 9, 12]) == 4
assert longest_arith_seq_length([9, 4, 7, 2, 10]) == 3
assert longest_arith_seq_length([20, 1, 15, 3, 10, 5, 8]) == 4
print(longest_arith_seq_length([20, 1, 15, 3, 10, 5, 8]))

---
# 3) Knapsack

**0/1 knapsack** (each item once): iterate capacity *downwards* so an item is not reused in
the same pass. **Unbounded knapsack** (reuse allowed): iterate capacity *upwards*. For
counting problems, the loop order also decides whether you count **combinations** (items
outer, capacity inner) or **ordered sequences** (capacity outer, items inner).

## 3.1 Subset sums and coins

### 3.1.1 [322. Coin Change](https://leetcode.com/problems/coin-change/) — ★★★ · Medium

**Problem.** Fewest coins summing to `amount` (unlimited coins), or −1.  
**Signal.** Unbounded knapsack minimising count.  
**Idea.** `dp[a] = 1 + min(dp[a − c])` over coins `c ≤ a`; `dp[0] = 0`. Greedy fails (e.g. coins `[1, 3, 4]`, amount 6).  
**Cost.** O(amount · coins) time · O(amount) space

In [ ]:
def coin_change(coins: list[int], amount: int) -> int:
    """Fewest coins for amount, or -1."""
    inf = amount + 1
    dp = [0] + [inf] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] < inf else -1


assert coin_change([1, 2, 5], 11) == 3
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
assert coin_change([1, 3, 4], 6) == 2
print(coin_change([1, 2, 5], 11))

### 3.1.2 [518. Coin Change II](https://leetcode.com/problems/coin-change-ii/) — ★★ · Medium

**Problem.** Number of *combinations* of coins that make `amount`.  
**Signal.** Unbounded knapsack counting; coins in the outer loop so each combination counts once.  
**Idea.** For each coin, for `a` from `coin` to `amount`: `dp[a] += dp[a − coin]`. Because coins are processed one at a time, every combination is built in a fixed coin order.  
**Cost.** O(amount · coins)  
**Trap.** Swapping the loops counts ordered sequences instead (that is Combination Sum IV).

In [ ]:
def change(amount: int, coins: list[int]) -> int:
    """Number of coin combinations for amount."""
    dp = [1] + [0] * amount
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]


assert change(5, [1, 2, 5]) == 4
assert change(3, [2]) == 0
assert change(10, [10]) == 1
print(change(5, [1, 2, 5]))

### 3.1.3 [377. Combination Sum IV](https://leetcode.com/problems/combination-sum-iv/) — ★★★ · Medium

**Problem.** Number of *ordered* sequences of `nums` (reuse allowed) summing to `target`.  
**Signal.** Counting sequences → target in the outer loop.  
**Idea.** `dp[t] = Σ dp[t − x]` over `x ≤ t`: the last element of the sequence is any `x`. Order matters, so `(1, 2)` and `(2, 1)` are both counted.  
**Cost.** O(target · n)

In [ ]:
def combination_sum4(nums: list[int], target: int) -> int:
    """Ordered ways to sum to target."""
    dp = [1] + [0] * target
    for t in range(1, target + 1):
        dp[t] = sum(dp[t - x] for x in nums if x <= t)
    return dp[target]


assert combination_sum4([1, 2, 3], 4) == 7
assert combination_sum4([9], 3) == 0
print(combination_sum4([1, 2, 3], 4))

### 3.1.4 [416. Partition Equal Subset Sum](https://leetcode.com/problems/partition-equal-subset-sum/) — ★★ · Medium

**Problem.** Can the array be split into two subsets with equal sums?  
**Signal.** Subset sum to `total / 2` → 0/1 knapsack on booleans.  
**Idea.** If the total is odd, no. Otherwise maintain the set of reachable sums (or a boolean array iterated downwards). A Python int used as a bitset makes it one line per number: `reach |= reach << x`.  
**Cost.** O(n · total) time

In [ ]:
def can_partition(nums: list[int]) -> bool:
    """True if nums splits into two equal-sum halves."""
    total = sum(nums)
    if total % 2:
        return False
    half = total // 2
    dp = [True] + [False] * half
    for x in nums:
        for s in range(half, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    return dp[half]


assert can_partition([1, 5, 11, 5])
assert not can_partition([1, 2, 3, 5])
print(can_partition([1, 5, 11, 5]))

### 3.1.5 [494. Target Sum](https://leetcode.com/problems/target-sum/) — ★★ · Medium

**Problem.** Assign `+` or `−` to each number; count assignments evaluating to `target`.  
**Signal.** Split into positive set `P` and negative set `N`: `P = (total + target) / 2` → count subsets.  
**Idea.** `P − N = target` and `P + N = total` give `P = (total + target) / 2`. If that is not a non-negative integer, the answer is 0. Count subsets with sum `P` via 0/1 knapsack (capacity iterated downwards).  
**Cost.** O(n · total)

In [ ]:
def find_target_sum_ways(nums: list[int], target: int) -> int:
    """Number of +/- assignments reaching target."""
    total = sum(nums)
    if abs(target) > total or (total + target) % 2:
        return 0
    goal = (total + target) // 2
    dp = [1] + [0] * goal
    for x in nums:
        for s in range(goal, x - 1, -1):
            dp[s] += dp[s - x]
    return dp[goal]


assert find_target_sum_ways([1, 1, 1, 1, 1], 3) == 5
assert find_target_sum_ways([1], 1) == 1
assert find_target_sum_ways([0, 0, 1], 1) == 4
print(find_target_sum_ways([1, 1, 1, 1, 1], 3))

### 3.1.6 [1049. Last Stone Weight II](https://leetcode.com/problems/last-stone-weight-ii/) — ★ · Medium

**Problem.** Smash stones pairwise (result `|a − b|`); minimise the last remaining weight.  
**Signal.** The result is `|sum(A) − sum(B)|` for some split → subset sum closest to `total / 2`.  
**Idea.** Find the largest reachable subset sum `s ≤ total // 2`; the answer is `total − 2s`.  
**Cost.** O(n · total)

In [ ]:
def last_stone_weight_ii(stones: list[int]) -> int:
    """Smallest possible final stone weight."""
    total = sum(stones)
    reach = {0}
    for x in stones:
        reach |= {s + x for s in reach if s + x <= total // 2}
    return total - 2 * max(reach)


assert last_stone_weight_ii([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight_ii([31, 26, 33, 21, 40]) == 5
print(last_stone_weight_ii([2, 7, 4, 1, 8, 1]))

### 3.1.7 [474. Ones and Zeroes](https://leetcode.com/problems/ones-and-zeroes/) — ★ · Medium

**Problem.** Largest subset of binary strings using at most `m` zeros and `n` ones in total.  
**Signal.** 0/1 knapsack with two capacities.  
**Idea.** `dp[i][j]` = max strings using ≤ `i` zeros and ≤ `j` ones. For each string, iterate both capacities downwards: `dp[i][j] = max(dp[i][j], dp[i − z][j − o] + 1)`.  
**Cost.** O(L · m · n)

In [ ]:
def find_max_form(strs: list[str], m: int, n: int) -> int:
    """Largest subset within the zero/one budget."""
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        zeros, ones = s.count("0"), s.count("1")
        for i in range(m, zeros - 1, -1):
            for j in range(n, ones - 1, -1):
                dp[i][j] = max(dp[i][j], dp[i - zeros][j - ones] + 1)
    return dp[m][n]


assert find_max_form(["10", "0001", "111001", "1", "0"], 5, 3) == 4
assert find_max_form(["10", "0", "1"], 1, 1) == 2
print(find_max_form(["10", "0001", "111001", "1", "0"], 5, 3))

---
# 4) State Machine DP

When the choice at step `i` depends on a *mode* (holding a stock, in cooldown, number of
transactions used), add the mode to the state: `dp[i][mode]`. Draw the modes as boxes and
the allowed transitions as arrows; each arrow is one line of code.

## 4.1 Stock problems

### 4.1.1 [122. Best Time to Buy and Sell Stock II](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-ii/) — ★★ · Medium

**Problem.** Unlimited transactions (hold at most one share); maximise profit.  
**Signal.** Two states (holding / not holding) — or greedily take every upward move.  
**Idea.** Any profitable multi-day hold equals the sum of its daily increases, so add every positive `p[i] − p[i−1]`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_profit_ii(prices: list[int]) -> int:
    """Profit with unlimited transactions."""
    return sum(max(0, b - a) for a, b in zip(prices, prices[1:]))


assert max_profit_ii([7, 1, 5, 3, 6, 4]) == 7
assert max_profit_ii([1, 2, 3, 4, 5]) == 4
assert max_profit_ii([7, 6, 4, 3, 1]) == 0
print(max_profit_ii([7, 1, 5, 3, 6, 4]))

### 4.1.2 [309. Best Time to Buy and Sell Stock with Cooldown](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-cooldown/) — ★★ · Medium

**Problem.** Unlimited transactions, but after selling you must wait one day before buying.  
**Signal.** Three states: `hold`, `sold` (just sold today), `rest` (free to buy).  
**Idea.** `hold = max(hold, rest − p)`, `sold = hold + p`, `rest = max(rest, sold)` — using yesterday's values on the right-hand side. Buying only from `rest` enforces the cooldown.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_profit_cooldown(prices: list[int]) -> int:
    """Profit with a one-day cooldown after selling."""
    hold, sold, rest = float("-inf"), 0, 0
    for p in prices:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)


assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert max_profit_cooldown([1]) == 0
print(max_profit_cooldown([1, 2, 3, 0, 2]))

### 4.1.3 [714. Best Time to Buy and Sell Stock with Transaction Fee](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-transaction-fee/) — ★ · Medium

**Problem.** Unlimited transactions, each sale costs `fee`.  
**Signal.** Two states with the fee charged on selling.  
**Idea.** `cash = max(cash, hold + p − fee)`, `hold = max(hold, cash − p)`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_profit_fee(prices: list[int], fee: int) -> int:
    """Profit with a per-transaction fee."""
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        cash = max(cash, hold + p - fee)
        hold = max(hold, cash - p)
    return cash


assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert max_profit_fee([1, 3, 7, 5, 10, 3], 3) == 6
print(max_profit_fee([1, 3, 2, 8, 4, 9], 2))

### 4.1.4 [123. Best Time to Buy and Sell Stock III](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-iii/) — ★★ · Hard

**Problem.** At most two transactions.  
**Signal.** Four states: after first buy, first sell, second buy, second sell.  
**Idea.** `b1 = max(b1, −p)`, `s1 = max(s1, b1 + p)`, `b2 = max(b2, s1 − p)`, `s2 = max(s2, b2 + p)`. Each state is the best balance after that action so far.  
**Cost.** O(n) time · O(1) space

In [ ]:
def max_profit_iii(prices: list[int]) -> int:
    """Profit with at most two transactions."""
    b1 = b2 = float("-inf")
    s1 = s2 = 0
    for p in prices:
        b1 = max(b1, -p)
        s1 = max(s1, b1 + p)
        b2 = max(b2, s1 - p)
        s2 = max(s2, b2 + p)
    return s2


assert max_profit_iii([3, 3, 5, 0, 0, 3, 1, 4]) == 6
assert max_profit_iii([1, 2, 3, 4, 5]) == 4
assert max_profit_iii([7, 6, 4, 3, 1]) == 0
print(max_profit_iii([3, 3, 5, 0, 0, 3, 1, 4]))

### 4.1.5 [188. Best Time to Buy and Sell Stock IV](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-iv/) — ★★ · Hard

**Problem.** At most `k` transactions.  
**Signal.** Generalise the four-state version to arrays `buy[1..k]`, `sell[1..k]`.  
**Idea.** For each price, update `buy[j] = max(buy[j], sell[j−1] − p)` and `sell[j] = max(sell[j], buy[j] + p)` for `j = 1..k`. If `k ≥ n/2`, it is the unlimited case.  
**Cost.** O(n · k) time · O(k) space

In [ ]:
def max_profit_iv(k: int, prices: list[int]) -> int:
    """Profit with at most k transactions."""
    if k >= len(prices) // 2:
        return max_profit_ii(prices)
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for j in range(1, k + 1):
            buy[j] = max(buy[j], sell[j - 1] - p)
            sell[j] = max(sell[j], buy[j] + p)
    return sell[k]


assert max_profit_iv(2, [2, 4, 1]) == 2
assert max_profit_iv(2, [3, 2, 6, 5, 0, 3]) == 7
print(max_profit_iv(2, [3, 2, 6, 5, 0, 3]))

---
# 5) Grid DP

For paths that move only right/down, `dp[r][c]` depends on the cell above and the cell to
the left, so a single row array updated in place is enough.

## 5.1 Paths on grids

### 5.1.1 [62. Unique Paths](https://leetcode.com/problems/unique-paths/) — ★★★ · Medium

**Problem.** Robot moves only right or down in an `m × n` grid; number of paths from top-left to bottom-right.  
**Signal.** `dp[r][c] = dp[r−1][c] + dp[r][c−1]`.  
**Idea.** Keep one row: `row[c] += row[c−1]` adds the left neighbour to the value from above. (Closed form: `C(m+n−2, m−1)`.)  
**Cost.** O(mn) time · O(n) space

In [ ]:
def unique_paths(m: int, n: int) -> int:
    """Lattice paths in an m x n grid."""
    row = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            row[c] += row[c - 1]
    return row[-1]


assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
print(unique_paths(3, 7))

### 5.1.2 [63. Unique Paths II](https://leetcode.com/problems/unique-paths-ii/) — ★★ · Medium

**Problem.** Same with obstacles (`1` cells cannot be entered).  
**Signal.** Unique Paths with `dp = 0` on obstacles.  
**Idea.** Start with `row[0] = 1`; for each cell, an obstacle sets 0, otherwise add the left value.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def unique_paths_with_obstacles(grid: list[list[int]]) -> int:
    """Lattice paths avoiding obstacles."""
    row = [0] * len(grid[0])
    row[0] = 1
    for r in grid:
        for c, cell in enumerate(r):
            if cell:
                row[c] = 0
            elif c:
                row[c] += row[c - 1]
    return row[-1]


assert unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert unique_paths_with_obstacles([[0, 1], [0, 0]]) == 1
assert unique_paths_with_obstacles([[1]]) == 0
print(unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))

### 5.1.3 [64. Minimum Path Sum](https://leetcode.com/problems/minimum-path-sum/) — ★★ · Medium

**Problem.** Minimum sum path from top-left to bottom-right moving right/down.  
**Signal.** `dp[r][c] = grid[r][c] + min(up, left)`.  
**Idea.** One row array; the first row and column have only one predecessor.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def min_path_sum(grid: list[list[int]]) -> int:
    """Cheapest monotone path sum."""
    inf = float("inf")
    row = [inf] * len(grid[0])
    row[0] = 0
    for r in grid:
        for c, cell in enumerate(r):
            left = row[c - 1] if c else inf
            row[c] = cell + min(row[c], left)
    return row[-1]


assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

### 5.1.4 [120. Triangle](https://leetcode.com/problems/triangle/) — ★★ · Medium

**Problem.** Minimum path sum from top to bottom of a triangle (move to adjacent numbers below).  
**Signal.** Bottom-up: each cell picks the better of its two children.  
**Idea.** Start with the last row; for each row upward, `dp[c] = tri[r][c] + min(dp[c], dp[c+1])`. Bottom-up avoids edge cases at the triangle's borders.  
**Cost.** O(n²) time · O(n) space

In [ ]:
def minimum_total(triangle: list[list[int]]) -> int:
    """Min top-to-bottom path sum."""
    dp = triangle[-1].copy()
    for r in range(len(triangle) - 2, -1, -1):
        for c in range(r + 1):
            dp[c] = triangle[r][c] + min(dp[c], dp[c + 1])
    return dp[0]


assert minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
assert minimum_total([[-10]]) == -10
print(minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]))

### 5.1.5 [931. Minimum Falling Path Sum](https://leetcode.com/problems/minimum-falling-path-sum/) — ★ · Medium

**Problem.** Path from any top cell to any bottom cell, moving down, down-left or down-right; minimum sum?  
**Signal.** Each cell adds the best of three cells above.  
**Idea.** Row by row: `new[c] = m[r][c] + min(prev[c−1], prev[c], prev[c+1])` with bounds checks.  
**Cost.** O(n²) time · O(n) space

In [ ]:
def min_falling_path_sum(matrix: list[list[int]]) -> int:
    """Min falling path sum."""
    prev = matrix[0]
    n = len(prev)
    for row in matrix[1:]:
        prev = [row[c] + min(prev[max(0, c - 1) : min(n, c + 2)]) for c in range(n)]
    return min(prev)


assert min_falling_path_sum([[2, 1, 3], [6, 5, 4], [7, 8, 9]]) == 13
assert min_falling_path_sum([[-19, 57], [-40, -5]]) == -59
print(min_falling_path_sum([[2, 1, 3], [6, 5, 4], [7, 8, 9]]))

### 5.1.6 [221. Maximal Square](https://leetcode.com/problems/maximal-square/) — ★★ · Medium

**Problem.** Area of the largest square of `"1"`s.  
**Signal.** `dp[r][c]` = side of the largest square with bottom-right corner at `(r, c)`.  
**Idea.** A square of side `k` ending at `(r, c)` needs squares of side `k − 1` ending above, left and diagonally: `dp = 1 + min(up, left, diag)` for a `"1"` cell.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def maximal_square(matrix: list[list[str]]) -> int:
    """Area of the largest all-ones square."""
    cols = len(matrix[0])
    dp = [0] * (cols + 1)
    best = 0
    for row in matrix:
        diag = 0
        for c in range(1, cols + 1):
            above = dp[c]
            dp[c] = 1 + min(dp[c], dp[c - 1], diag) if row[c - 1] == "1" else 0
            diag = above
            best = max(best, dp[c])
    return best * best


m = [list("10100"), list("10111"), list("11111"), list("10010")]
assert maximal_square(m) == 4
assert maximal_square([list("01"), list("10")]) == 1
print(maximal_square(m))

### 5.1.7 [174. Dungeon Game](https://leetcode.com/problems/dungeon-game/) — ★ · Hard

**Problem.** Knight moves right/down from top-left to bottom-right; cells add or subtract health; health must stay ≥ 1. Minimum starting health?  
**Signal.** Forward DP can't decide; go **backwards** from the princess.  
**Idea.** `need[r][c]` = minimum health needed when entering `(r, c)`: `max(1, min(need_right, need_down) − dungeon[r][c])`. Fill from the bottom-right corner.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def calculate_minimum_hp(dungeon: list[list[int]]) -> int:
    """Minimum initial health to survive."""
    rows, cols = len(dungeon), len(dungeon[0])
    inf = float("inf")
    need = [inf] * (cols + 1)
    need[cols - 1] = 1
    for r in range(rows - 1, -1, -1):
        nxt = [inf] * (cols + 1)
        for c in range(cols - 1, -1, -1):
            best = min(need[c], nxt[c + 1])
            nxt[c] = max(1, best - dungeon[r][c])
        need = nxt
    return need[0]


assert calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7
assert calculate_minimum_hp([[0]]) == 1
print(calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]))

### 5.1.8 [329. Longest Increasing Path in a Matrix](https://leetcode.com/problems/longest-increasing-path-in-a-matrix/) — ★★ · Hard

**Problem.** Length of the longest strictly increasing path (4 directions) in a matrix.  
**Signal.** DFS + memo: the increasing constraint makes the graph acyclic.  
**Idea.** `longest(r, c) = 1 + max(longest(neighbour))` over larger neighbours. No visited set is needed because a path can never return to a smaller value. Memoise each cell.  
**Cost.** O(mn) time · O(mn) space

In [ ]:
def longest_increasing_path(matrix: list[list[int]]) -> int:
    """Longest strictly increasing path length."""
    rows, cols = len(matrix), len(matrix[0])

    @cache
    def longest(r: int, c: int) -> int:
        best = 1
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + longest(nr, nc))
        return best

    return max(longest(r, c) for r in range(rows) for c in range(cols))


assert longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]) == 4
assert longest_increasing_path([[3, 4, 5], [3, 2, 6], [2, 2, 1]]) == 4
print(longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]))

---
# 6) String DP

**Two strings:** `dp[i][j]` describes the prefixes `a[:i]` and `b[:j]`. The transition looks
at the last characters `a[i−1]`, `b[j−1]`: if they match, extend `dp[i−1][j−1]`;
otherwise combine `dp[i−1][j]` and `dp[i][j−1]`. Row `0` / column `0` are the
empty-prefix base cases. **One string, palindromes:** `dp[l][r]` over intervals, filled by
increasing length (or expand around centres).

## 6.1 Palindromes

### 6.1.1 [5. Longest Palindromic Substring](https://leetcode.com/problems/longest-palindromic-substring/) — ★★★ · Medium

**Problem.** Longest palindromic substring.  
**Signal.** Every palindrome has a centre → expand around 2n − 1 centres.  
**Idea.** For each centre (a character or the gap between two), expand while both ends match. Track the longest span. Simpler and lighter than an O(n²) table.  
**Cost.** O(n²) time · O(1) space

In [ ]:
def longest_palindrome_substring(s: str) -> str:
    """Longest palindromic substring."""
    best = (0, 0)
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        if hi - lo - 1 > best[1] - best[0]:
            best = (lo + 1, hi)
    return s[best[0] : best[1]]


assert longest_palindrome_substring("babad") in ("bab", "aba")
assert longest_palindrome_substring("cbbd") == "bb"
assert longest_palindrome_substring("a") == "a"
print(longest_palindrome_substring("forgeeksskeegfor"))

### 6.1.2 [647. Palindromic Substrings](https://leetcode.com/problems/palindromic-substrings/) — ★★★ · Medium

**Problem.** Count palindromic substrings.  
**Signal.** Expand around centres, counting every successful expansion.  
**Idea.** Each successful expansion step is one more palindrome.  
**Cost.** O(n²) time · O(1) space

In [ ]:
def count_substrings(s: str) -> int:
    """Number of palindromic substrings."""
    count = 0
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            count += 1
            lo, hi = lo - 1, hi + 1
    return count


assert count_substrings("abc") == 3
assert count_substrings("aaa") == 6
print(count_substrings("aaa"))

### 6.1.3 [516. Longest Palindromic Subsequence](https://leetcode.com/problems/longest-palindromic-subsequence/) — ★ · Medium

**Problem.** Length of the longest palindromic subsequence.  
**Signal.** Interval DP `dp[l][r]` — or LCS of `s` and `reversed(s)`.  
**Idea.** If `s[l] == s[r]`, `dp[l][r] = dp[l+1][r−1] + 2`; else `max(dp[l+1][r], dp[l][r−1])`. Fill `l` from right to left so `dp[l+1]` is ready.  
**Cost.** O(n²) time · O(n) space

In [ ]:
def longest_palindrome_subseq(s: str) -> int:
    """Longest palindromic subsequence length."""
    n = len(s)
    dp = [0] * n
    for l in range(n - 1, -1, -1):
        prev_diag = 0
        dp[l] = 1
        for r in range(l + 1, n):
            saved = dp[r]
            if s[l] == s[r]:
                dp[r] = prev_diag + 2
            else:
                dp[r] = max(dp[r], dp[r - 1])
            prev_diag = saved
    return dp[-1]


assert longest_palindrome_subseq("bbbab") == 4
assert longest_palindrome_subseq("cbbd") == 2
print(longest_palindrome_subseq("bbbab"))

### 6.1.4 [132. Palindrome Partitioning II](https://leetcode.com/problems/palindrome-partitioning-ii/) — ★ · Hard

**Problem.** Minimum cuts so every piece is a palindrome.  
**Signal.** `cuts[i]` = min cuts for `s[:i+1]`, with centre expansion to find palindromes.  
**Idea.** Expanding around each centre finds every palindrome `s[lo..hi]`; each one gives `cuts[hi] = min(cuts[hi], cuts[lo−1] + 1)` (or 0 when `lo == 0`).  
**Cost.** O(n²) time · O(n) space

In [ ]:
def min_cut(s: str) -> int:
    """Fewest cuts into palindromes."""
    n = len(s)
    cuts = list(range(n))
    for centre in range(2 * n - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        while lo >= 0 and hi < n and s[lo] == s[hi]:
            cuts[hi] = min(cuts[hi], 0 if lo == 0 else cuts[lo - 1] + 1)
            lo, hi = lo - 1, hi + 1
    return cuts[-1]


assert min_cut("aab") == 1
assert min_cut("a") == 0
assert min_cut("ab") == 1
print(min_cut("aab"))

## 6.2 Two-string alignment

### 6.2.1 [1143. Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest common subsequence of two strings.  
**Signal.** The canonical two-string DP.  
**Idea.** If `a[i−1] == b[j−1]`, `dp[i][j] = dp[i−1][j−1] + 1`; else `max(dp[i−1][j], dp[i][j−1])`. Keep one row plus the diagonal value.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def longest_common_subsequence(a: str, b: str) -> int:
    """LCS length."""
    dp = [0] * (len(b) + 1)
    for ch in a:
        diag = 0
        for j in range(1, len(b) + 1):
            above = dp[j]
            dp[j] = diag + 1 if ch == b[j - 1] else max(dp[j], dp[j - 1])
            diag = above
    return dp[-1]


assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "def") == 0
print(longest_common_subsequence("abcde", "ace"))

### 6.2.2 [583. Delete Operation for Two Strings](https://leetcode.com/problems/delete-operation-for-two-strings/) — ★ · Medium

**Problem.** Fewest deletions (from either string) to make them equal.  
**Signal.** Keep the LCS, delete everything else.  
**Idea.** Answer `len(a) + len(b) − 2 × LCS(a, b)`.  
**Cost.** O(mn)

In [ ]:
def min_distance_delete(a: str, b: str) -> int:
    """Deletions to make two strings equal."""
    return len(a) + len(b) - 2 * longest_common_subsequence(a, b)


assert min_distance_delete("sea", "eat") == 2
assert min_distance_delete("leetcode", "etco") == 4
print(min_distance_delete("sea", "eat"))

### 6.2.3 [72. Edit Distance](https://leetcode.com/problems/edit-distance/) — ★★ · Medium

**Problem.** Fewest inserts, deletes and replaces to turn `a` into `b`.  
**Signal.** Two-string DP with three operations.  
**Idea.** `dp[i][j]` = edits for `a[:i]` → `b[:j]`. Matching last characters cost nothing (`dp[i−1][j−1]`); otherwise `1 + min(replace dp[i−1][j−1], delete dp[i−1][j], insert dp[i][j−1])`. Base cases: `dp[i][0] = i`, `dp[0][j] = j`.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def min_distance(a: str, b: str) -> int:
    """Levenshtein distance."""
    dp = list(range(len(b) + 1))
    for i, ch in enumerate(a, 1):
        diag, dp[0] = dp[0], i
        for j in range(1, len(b) + 1):
            above = dp[j]
            if ch == b[j - 1]:
                dp[j] = diag
            else:
                dp[j] = 1 + min(diag, above, dp[j - 1])
            diag = above
    return dp[-1]


assert min_distance("horse", "ros") == 3
assert min_distance("intention", "execution") == 5
assert min_distance("", "abc") == 3
print(min_distance("horse", "ros"))

### 6.2.4 [97. Interleaving String](https://leetcode.com/problems/interleaving-string/) — ★★ · Medium

**Problem.** Is `s3` an interleaving of `s1` and `s2` (keeping each one's order)?  
**Signal.** `dp[i][j]` = `s3[:i+j]` is an interleaving of `s1[:i]` and `s2[:j]`.  
**Idea.** The last character of `s3[:i+j]` came from `s1` (needs `dp[i−1][j]` and `s1[i−1] == s3[i+j−1]`) or from `s2` (needs `dp[i][j−1]` and a matching `s2[j−1]`).  
**Cost.** O(mn) time · O(n) space

In [ ]:
def is_interleave(s1: str, s2: str, s3: str) -> bool:
    """True if s3 interleaves s1 and s2."""
    if len(s1) + len(s2) != len(s3):
        return False
    dp = [True] + [False] * len(s2)
    for j in range(1, len(s2) + 1):
        dp[j] = dp[j - 1] and s2[j - 1] == s3[j - 1]
    for i in range(1, len(s1) + 1):
        dp[0] = dp[0] and s1[i - 1] == s3[i - 1]
        for j in range(1, len(s2) + 1):
            dp[j] = (dp[j] and s1[i - 1] == s3[i + j - 1]) or (
                dp[j - 1] and s2[j - 1] == s3[i + j - 1]
            )
    return dp[-1]


assert is_interleave("aabcc", "dbbca", "aadbbcbcac")
assert not is_interleave("aabcc", "dbbca", "aadbbbaccc")
assert is_interleave("", "", "")
print(is_interleave("aabcc", "dbbca", "aadbbcbcac"))

### 6.2.5 [115. Distinct Subsequences](https://leetcode.com/problems/distinct-subsequences/) — ★★ · Hard

**Problem.** Number of distinct subsequences of `s` equal to `t`.  
**Signal.** Counting two-string DP.  
**Idea.** `dp[i][j]` = ways `s[:i]` contains `t[:j]`. Always `dp[i−1][j]` (skip `s[i−1]`); if `s[i−1] == t[j−1]`, also add `dp[i−1][j−1]` (use it). One row iterated with `j` descending.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def num_distinct(s: str, t: str) -> int:
    """Ways to form t as a subsequence of s."""
    dp = [1] + [0] * len(t)
    for ch in s:
        for j in range(len(t), 0, -1):
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[-1]


assert num_distinct("rabbbit", "rabbit") == 3
assert num_distinct("babgbag", "bag") == 5
print(num_distinct("babgbag", "bag"))

### 6.2.6 [10. Regular Expression Matching](https://leetcode.com/problems/regular-expression-matching/) — ★★ · Hard

**Problem.** Match `s` against pattern `p` with `.` (any char) and `*` (zero or more of the previous).  
**Signal.** Two-string DP; `*` either uses zero copies or consumes one character of `s`.  
**Idea.** `match(i, j)` = does `s[i:]` match `p[j:]`. If `p[j+1] == '*'`: skip `x*` (`match(i, j+2)`) or, if the first character matches, consume it (`match(i+1, j)`). Otherwise match one character and move both. Memoise.  
**Cost.** O(mn) time and space

In [ ]:
def is_match_regex(s: str, p: str) -> bool:
    """Regex match supporting . and *."""

    @cache
    def match(i: int, j: int) -> bool:
        if j == len(p):
            return i == len(s)
        first = i < len(s) and p[j] in (s[i], ".")
        if j + 1 < len(p) and p[j + 1] == "*":
            return match(i, j + 2) or (first and match(i + 1, j))
        return first and match(i + 1, j + 1)

    return match(0, 0)


assert not is_match_regex("aa", "a")
assert is_match_regex("aa", "a*")
assert is_match_regex("ab", ".*")
assert is_match_regex("aab", "c*a*b")
assert not is_match_regex("mississippi", "mis*is*p*.")
print(is_match_regex("aab", "c*a*b"))

### 6.2.7 [44. Wildcard Matching](https://leetcode.com/problems/wildcard-matching/) — ★ · Hard

**Problem.** Match with `?` (any single char) and `*` (any sequence, including empty).  
**Signal.** Two-string DP; `*` either matches empty or eats one more character.  
**Idea.** `dp[i][j]` = `s[:i]` matches `p[:j]`. For `*`: `dp[i][j−1]` (empty) or `dp[i−1][j]` (consume `s[i−1]`). For `?` or equal chars: `dp[i−1][j−1]`.  
**Cost.** O(mn) time · O(n) space

In [ ]:
def is_match_wildcard(s: str, p: str) -> bool:
    """Wildcard match supporting ? and *."""
    dp = [True] + [False] * len(p)
    for j in range(1, len(p) + 1):
        dp[j] = dp[j - 1] and p[j - 1] == "*"
    for ch in s:
        diag, dp[0] = dp[0], False
        for j in range(1, len(p) + 1):
            above = dp[j]
            if p[j - 1] == "*":
                dp[j] = dp[j] or dp[j - 1]
            else:
                dp[j] = diag and p[j - 1] in (ch, "?")
            diag = above
    return dp[-1]


assert not is_match_wildcard("aa", "a")
assert is_match_wildcard("aa", "*")
assert not is_match_wildcard("cb", "?a")
assert is_match_wildcard("adceb", "*a*b")
print(is_match_wildcard("adceb", "*a*b"))

---
# 7) Interval and Game DP

`dp[l][r]` describes the subarray `l..r`. Fill by increasing interval length so smaller
intervals are ready. A useful trick: choose the element handled **last** in the interval
(not first) — then the two sides become independent subproblems.

## 7.1 Interval DP

### 7.1.1 [312. Burst Balloons](https://leetcode.com/problems/burst-balloons/) — ★★ · Hard

**Problem.** Bursting balloon `i` earns `left × nums[i] × right` with its current neighbours. Maximise coins.  
**Signal.** Choose the balloon burst **last** in each interval — its neighbours are then fixed.  
**Idea.** Pad with 1s. `dp[l][r]` = best coins from bursting everything strictly between `l` and `r`. If `k` is last, its neighbours are `l` and `r`: `dp[l][r] = max(dp[l][k] + nums[l]·nums[k]·nums[r] + dp[k][r])`.  
**Cost.** O(n³) time · O(n²) space

In [ ]:
def max_coins(nums: list[int]) -> int:
    """Max coins from bursting all balloons."""
    vals = [1] + nums + [1]
    n = len(vals)
    dp = [[0] * n for _ in range(n)]
    for length in range(2, n):
        for l in range(n - length):
            r = l + length
            dp[l][r] = max(
                dp[l][k] + vals[l] * vals[k] * vals[r] + dp[k][r]
                for k in range(l + 1, r)
            )
    return dp[0][n - 1]


assert max_coins([3, 1, 5, 8]) == 167
assert max_coins([1, 5]) == 10
print(max_coins([3, 1, 5, 8]))

### 7.1.2 [1547. Minimum Cost to Cut a Stick](https://leetcode.com/problems/minimum-cost-to-cut-a-stick/) — ★ · Hard

**Problem.** Cut a stick of length `n` at given positions; each cut costs the current piece's length. Minimise total cost.  
**Signal.** Burst Balloons in disguise: the first cut in an interval splits it into two independent intervals.  
**Idea.** Sort cut positions and add `0` and `n`. `dp[i][j]` = min cost to make all cuts strictly between `cuts[i]` and `cuts[j]`: `cuts[j] − cuts[i] + min(dp[i][k] + dp[k][j])`.  
**Cost.** O(m³) for m cuts

In [ ]:
def min_cost_cut(n: int, cuts: list[int]) -> int:
    """Cheapest order of cutting the stick."""
    pos = [0] + sorted(cuts) + [n]

    @cache
    def cost(i: int, j: int) -> int:
        if j - i < 2:
            return 0
        return pos[j] - pos[i] + min(cost(i, k) + cost(k, j) for k in range(i + 1, j))

    return cost(0, len(pos) - 1)


assert min_cost_cut(7, [1, 3, 4, 5]) == 16
assert min_cost_cut(9, [5, 6, 1, 4, 2]) == 22
print(min_cost_cut(7, [1, 3, 4, 5]))

### 7.1.3 [486. Predict the Winner](https://leetcode.com/problems/predict-the-winner/) — ★ · Medium

**Problem.** Players alternately take from either end of the array; can player 1 win (ties count)?  
**Signal.** Minimax as a score *difference*: `diff(l, r)` = current player's best margin.  
**Idea.** `diff(l, r) = max(nums[l] − diff(l+1, r), nums[r] − diff(l, r−1))` — after you take, the opponent plays optimally on the rest, so subtract their margin. Player 1 wins iff `diff(0, n−1) ≥ 0`.  
**Cost.** O(n²)

In [ ]:
def predict_the_winner(nums: list[int]) -> bool:
    """True if player 1 can at least tie."""

    @cache
    def diff(l: int, r: int) -> int:
        if l == r:
            return nums[l]
        return max(nums[l] - diff(l + 1, r), nums[r] - diff(l, r - 1))

    return diff(0, len(nums) - 1) >= 0


assert not predict_the_winner([1, 5, 2])
assert predict_the_winner([1, 5, 233, 7])
print(predict_the_winner([1, 5, 233, 7]))

### 7.1.4 [877. Stone Game](https://leetcode.com/problems/stone-game/) — ★ · Medium

**Problem.** Even number of piles, odd total; players take from either end. Does Alice win?  
**Signal.** Same margin DP — and a parity argument says the first player always wins.  
**Idea.** Alice can take all even-indexed piles or all odd-indexed piles (she controls the parity of what Bob can reach), and one of those sets is strictly larger. The DP from Predict the Winner confirms it.  
**Cost.** O(1) by argument, O(n²) by DP

In [ ]:
def stone_game(piles: list[int]) -> bool:
    """True if Alice wins (always, for even pile counts)."""
    return predict_the_winner(piles)


assert stone_game([5, 3, 4, 5])
assert stone_game([3, 7, 2, 3])
print(stone_game([5, 3, 4, 5]))

### 7.1.5 [650. 2 Keys Keyboard](https://leetcode.com/problems/2-keys-keyboard/) — ★ · Medium

**Problem.** Start with one `A`; operations Copy All and Paste. Fewest operations to get `n` `A`s?  
**Signal.** Building `n` from a divisor `d` costs `n / d` operations (1 copy + `n/d − 1` pastes) → sum of prime factors.  
**Idea.** `dp[n] = min(dp[d] + n / d)` over divisors `d`. Equivalently, the answer is the sum of the prime factors of `n` with multiplicity.  
**Cost.** O(√n) by factorisation

In [ ]:
def min_steps(n: int) -> int:
    """Fewest copy/paste operations to reach n characters."""
    total, d = 0, 2
    while n > 1:
        while n % d == 0:
            total += d
            n //= d
        d += 1
    return total


assert min_steps(3) == 3
assert min_steps(1) == 0
assert min_steps(12) == 7
print(min_steps(12))

## 7.2 DP with binary search

### 7.2.1 [1235. Maximum Profit in Job Scheduling](https://leetcode.com/problems/maximum-profit-in-job-scheduling/) — ★★ · Hard

**Problem.** Jobs `(start, end, profit)`; choose non-overlapping jobs maximising profit.  
**Signal.** Weighted interval scheduling: sort by end, DP + binary search for the last compatible job.  
**Idea.** After sorting by end time, `dp[i]` = best using the first `i` jobs: `max(dp[i−1], profit_i + dp[k])`, where `k` = number of jobs ending at or before job `i`'s start (found with `bisect_right` on end times).  
**Cost.** O(n log n)

In [ ]:
def job_scheduling(start: list[int], end: list[int], profit: list[int]) -> int:
    """Max profit from non-overlapping jobs."""
    jobs = sorted(zip(end, start, profit))
    ends = [e for e, _, _ in jobs]
    dp = [0] * (len(jobs) + 1)
    for i, (_, s, p) in enumerate(jobs, 1):
        k = bisect_right(ends, s, hi=i - 1)
        dp[i] = max(dp[i - 1], p + dp[k])
    return dp[-1]


assert job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]) == 120
assert job_scheduling([1, 2, 3, 4, 6], [3, 5, 10, 6, 9], [20, 20, 100, 70, 60]) == 150
assert job_scheduling([1, 1, 1], [2, 3, 4], [5, 6, 4]) == 6
print(job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]))

---
# 8) Conclusion

| Family | Recognise it by | State → transition | Cost |
|---|---|---|---|
| Fibonacci-style | ways / best up to i, adjacency ban | `dp[i]` from `dp[i−1]`, `dp[i−2]` | O(n), O(1) |
| Kadane | best contiguous subarray (sum, product, circular) | best ending at i: extend or restart | O(n) |
| Word / partition | segment a string, split a number | `dp[i]` over prefix lengths | O(n·L) |
| LIS | increasing / chained subsequence | patience `tails` + binary search | O(n log n) |
| 0/1 knapsack | subset sum, partition, target sum | capacity loop **downwards** | O(n·W) |
| Unbounded knapsack | coins, squares | capacity loop **upwards**; loop order = combos vs sequences | O(n·W) |
| State machine | stock buy/sell with limits | one variable per state, yesterday's values | O(n·k) |
| Grid | right/down paths | one row array, in-place update | O(mn), O(n) |
| Backward grid | requirement depends on the future | fill from the end | O(mn) |
| Memo DFS on DAG | longest increasing path | `@cache` on cell | O(mn) |
| Palindromes | substrings / subsequences / cuts | expand around centres; `dp[l][r]` | O(n²) |
| Two strings | LCS, edit distance, interleave, regex | `dp[i][j]` over prefixes | O(mn) |
| Interval | burst balloons, cut stick, games | choose the **last** action in `[l, r]` | O(n³) |
| Weighted intervals | job scheduling | sort by end + bisect | O(n log n) |